# CCA-F Review Notebook — Tool Design MCP

This notebook follows the CCA-F Domain 2 task list. Each task includes a concept explanation, runnable Python example, exam summary, and two official-style practice questions.


## Environment Setup

In [ ]:
import json
import os
import re
from dataclasses import dataclass
from typing import Any

print("Environment ready: these examples are local simulations and require no API keys or external services.")


## D2.1 — Design MCP servers with tools, resources, and prompts

### 📖 Concept Explanation

An MCP server should not expose every possible capability as a tool. It should separate capabilities into three primitives: `tools` perform actions and may affect external state, `resources` provide read-only context, and `prompts` package reusable prompt templates. CCA-F questions often test whether you can place behavior at the correct boundary: read-only information should not be modeled as an action tool, and action tools need explicit input schemas, authorization boundaries, and failure semantics.

Tool failures should return structured results such as `isError=true`, an error category, a readable message, and retryability metadata. An empty result is different from a failed result. Transport also depends on deployment: local editor or CLI sidecar processes commonly use `stdio`, while shared remote services should use SSE or streamable HTTP with authentication, concurrency handling, and network failure behavior.


In [ ]:
# Task 2.1: MCP primitives, isError, and transport boundaries.
# This local dictionary simulates an MCP server without starting a real service.

mcp_server = {
    "name": "repo-review-server",
    "transport": "stdio",  # A local agent-spawned child process commonly uses stdio.
    "tools": {
        "run_unit_tests": {
            "description": "Run an allow-listed test command. Input: {'command': 'pytest tests/unit'}. Return isError=true on failure.",
            "mutates_state": False,
        },
        "create_issue": {
            "description": "Create an issue in the tracker. Input must include title and severity; not for querying existing issues.",
            "mutates_state": True,
        },
    },
    "resources": {
        "repo://architecture": "Read-only architecture notes; useful context, not an action.",
        "repo://coding-standards": "Read-only team coding standards.",
    },
    "prompts": {
        "api_review": "Review an API handler for auth, validation, error handling, and observability.",
    },
}

ALLOWED_TEST_COMMANDS = {"pytest tests/unit", "pytest tests/mcp"}

def mcp_result(content: Any, *, is_error: bool = False, category: str | None = None, retryable: bool = False) -> dict[str, Any]:
    """Build a tool result where errors, empty results, and successful results are distinct."""
    result = {"isError": is_error, "content": content}
    if is_error:
        result["error"] = {"category": category or "unknown", "retryable": retryable}
    return result


def run_unit_tests(args: dict[str, str]) -> dict[str, Any]:
    command = args.get("command", "")
    if command not in ALLOWED_TEST_COMMANDS:
        return mcp_result(
            {"message": "Test command is not allow-listed", "allowed": sorted(ALLOWED_TEST_COMMANDS)},
            is_error=True,
            category="policy_violation",
            retryable=False,
        )
    return mcp_result({"command": command, "passed": True, "tests": 12})


def choose_transport(location: str, shared_by_team: bool) -> str:
    # Exam point: stdio fits local processes; remote shared services need HTTP/SSE-style transport and auth boundaries.
    if location == "local" and not shared_by_team:
        return "stdio"
    return "streamable_http_or_sse"

print(json.dumps(mcp_server, indent=2))
print("Successful call:", run_unit_tests({"command": "pytest tests/unit"}))
print("Boundary error:", run_unit_tests({"command": "rm -rf /"}))
print("Remote team service transport:", choose_transport("remote", shared_by_team=True))


### ⚠️ Exam Summary & Common Traps

**✅ Correct patterns**

- Use `tools` for callable actions, `resources` for read-only context, and `prompts` for reusable templates.
- Return `isError=true` and structured error metadata for failures; keep empty successful results distinct.
- Put authorization, input schema, idempotency, and side effects in the tool boundary.
- Prefer `stdio` for local sidecar processes; use SSE/HTTP with auth, concurrency, and network failure handling for remote services.

**✗ Common traps**

- Turning documents, configuration, and templates into tools, which suggests they perform actions.
- Returning a plain string like `failed`, leaving the coordinator unable to decide whether to retry.
- Describing a state-changing tool as a lookup tool.
- Treating transport choice as syntax instead of a deployment boundary decision.


### 🧪 Practice Questions

**Q1.** An MCP tool times out, and the coordinator must decide whether retrying is safe. What should the tool return?

A) `isError=true` with structured fields such as `category`, `message`, and `retryable`  
B) An empty array, because no data was returned  
C) A plain string such as `timeout`, leaving the model to infer the meaning  
D) A hard process exit from the entire agent runtime

> **Answer: A**  
> **Explanation:** Structured errors distinguish failure from empty success and expose whether retrying is appropriate.

**Q2.** A team wants to move an MCP server from a local CLI workflow to a shared remote service. Which choice best matches CCA-F judgment?

A) Keep using only `stdio`, because the MCP primitives are unchanged  
B) Use SSE or streamable HTTP and add authentication, concurrency, and network error handling  
C) Convert all resources into tools so they are easier to call remotely  
D) Remove `isError` to avoid exposing internal failure details

> **Answer: B**  
> **Explanation:** Transport should match the deployment boundary; shared remote services need network and auth semantics.


## D2.2 — Configure tool distribution and manage tool selection reliability

### 📖 Concept Explanation

Tool-selection reliability starts with tool-set design, not with a longer system prompt. An agent should usually receive only four or five tools that are highly relevant to its job. The more tools it has, especially similarly named tools with broad descriptions, the more likely it is to choose incorrectly. Tool descriptions should include input format, when-to-use guidance, when-not-to-use boundaries, and important edge cases.

When tools overlap, disambiguate through names and descriptions first, for example `lookup_order_by_id` versus `search_orders_by_customer`. Do not rely on the model to guess from vague names. High-risk tools should also have programmatic guardrails such as schemas, allow lists, confirmation steps, or escalation.


In [ ]:
# Task 2.2: tool distribution and tool-selection reliability.
# This example shows small tool sets, explicit descriptions, edge cases, and overlap disambiguation.

agent_tools = {
    "refund_agent": [
        "verify_customer_identity",
        "lookup_order_by_id",
        "calculate_refund_quote",
        "submit_refund_request",
        "escalate_refund_case",
    ],
    "catalog_agent": [
        "search_products",
        "get_product_by_sku",
        "compare_product_specs",
        "check_inventory_by_sku",
    ],
}

tool_catalog = {
    "verify_customer_identity": "Input: email or customer_id. Verifies identity only; does not retrieve orders.",
    "lookup_order_by_id": "Input: order_id such as ORD-12345. Retrieves one order only; does not search by customer.",
    "search_orders_by_customer": "Input: customer_id. Returns a customer's order list; does not verify identity.",
    "submit_refund_request": "Input: order_id, amount, and reason. Creates a refund request; high-risk action.",
}

def candidate_tools(agent_name: str) -> list[str]:
    tools = agent_tools[agent_name]
    if len(tools) > 5:
        raise ValueError(f"{agent_name} has {len(tools)} tools. Split the agent or narrow its responsibility.")
    return tools


def choose_refund_tool(user_request: str) -> str:
    text = user_request.lower()
    has_order_id = bool(re.search(r"\bord-\d+\b", text))
    has_email = bool(re.search(r"[\w.-]+@[\w.-]+", text))

    if "refund" in text:
        if has_order_id:
            return "calculate_refund_quote"  # Quote and validate before submitting a high-risk action.
        return "ask_for_order_id"
    if has_order_id:
        return "lookup_order_by_id"
    if has_email or "customer" in text:
        return "verify_customer_identity"
    return "ask_clarifying_question"

examples = [
    "Check order ORD-12345",
    "Verify ada@example.com",
    "I need a refund for ORD-88888",
    "Find this customer's orders",
]

print("refund_agent tools:", candidate_tools("refund_agent"))
for request in examples:
    print(f"{request} => {choose_refund_tool(request)}")


### ⚠️ Exam Summary & Common Traps

**✅ Correct patterns**

- Keep each agent to roughly four or five highly relevant tools, then expand only when the responsibility is still clear.
- Use names that reveal input and purpose, such as `lookup_order_by_id`, instead of `get_data`.
- Document input format, when to use the tool, when not to use it, edge cases, and side effects.
- Disambiguate overlapping tools with names, descriptions, and schemas; add confirmation or escalation for high-risk actions.

**✗ Common traps**

- Exposing every tool to every agent, making the selection space too large.
- Giving multiple tools vague names such as `lookup` or descriptions like `gets information`.
- Adding many few-shot examples before fixing the tool boundary.
- Letting the model decide unaided whether to perform state-changing actions such as refunds, deletes, or sends.


### 🧪 Practice Questions

**Q1.** An agent often confuses `lookup_order_by_id` with `search_orders_by_customer`. What should you fix first?

A) Improve the tool names and descriptions, including input formats, use cases, and non-use cases  
B) Merge both tools into `get_information`  
C) Give the agent more database tools so it has broader coverage  
D) Increase temperature so the model explores more options

> **Answer: A**  
> **Explanation:** Tool confusion is usually addressed first by clarifying boundaries and descriptions; merging or adding tools increases ambiguity.

**Q2.** Which statement about tool count per agent best matches CCA-F style?

A) Provide 20 or more tools whenever possible so the agent is never blocked  
B) Usually keep four or five highly relevant tools and split responsibilities when the set grows too broad  
C) Tool count does not matter if the system prompt is long enough  
D) Keep one universal tool and describe all routing logic in natural language

> **Answer: B**  
> **Explanation:** Small, coherent tool sets improve selection reliability; broad responsibilities should be split or redesigned.


## D2.3 — Configure MCP in .mcp.json and manage environment variables

### 📖 Concept Explanation

A project-level `.mcp.json` registers MCP servers that a team can share and review in the repository. Personal experiments or machine-specific settings belong in user scope. A good configuration makes the server name, `command`, `args`, `env`, and runtime scope explicit. CCA-F often tests whether you understand that registration scope controls who can use the server, and environment variables control how secrets are injected locally, in CI/CD, and in deployment.

Secrets should not be hard-coded in `.mcp.json`. Prefer `${ENV_VAR}` references and provide real values through a CI/CD secret store, a developer shell, or the deployment platform. Configuration validation should check JSON shape, required fields, environment-variable references, appropriate scope, and whether CI has the required variables.


In [ ]:
# Task 2.3: .mcp.json, environment variables, scope, and CI/CD.
# This example validates configuration shape without reading real secrets.

mcp_json_text = """
{
  "mcpServers": {
    "orders": {
      "scope": "project",
      "command": "python",
      "args": ["-m", "company_mcp.orders"],
      "env": {
        "ORDERS_API_BASE": "https://api.example.com",
        "ORDERS_API_KEY": "${ORDERS_API_KEY}"
      }
    },
    "local_notes": {
      "scope": "user",
      "command": "python",
      "args": ["-m", "company_mcp.notes"],
      "env": {}
    }
  }
}
"""

ENV_REF = re.compile(r"^\$\{[A-Z][A-Z0-9_]*\}$")
SECRET_HINT = re.compile(r"(sk-|api[_-]?key|token)", re.IGNORECASE)

def validate_mcp_config(config: dict[str, Any], *, ci: bool = False) -> list[str]:
    errors: list[str] = []
    servers = config.get("mcpServers")
    if not isinstance(servers, dict) or not servers:
        return ["missing or invalid mcpServers"]

    for name, server in servers.items():
        if server.get("scope") not in {"project", "user"}:
            errors.append(f"{name}: scope must be project or user")
        if not server.get("command"):
            errors.append(f"{name}: missing command")
        if not isinstance(server.get("args", []), list):
            errors.append(f"{name}: args must be a list")
        for key, value in server.get("env", {}).items():
            value = str(value)
            if SECRET_HINT.search(key) and not ENV_REF.match(value):
                errors.append(f"{name}: {key} should use ${{ENV_VAR}}, not a hard-coded secret")
            if ci and ENV_REF.match(value):
                env_name = value[2:-1]
                if env_name not in os.environ:
                    errors.append(f"{name}: CI/CD is missing environment variable {env_name}")
    return errors

config = json.loads(mcp_json_text)
print(json.dumps(config, indent=2))
print("Local structural check:", validate_mcp_config(config))
print("CI/CD check:", validate_mcp_config(config, ci=True))


### ⚠️ Exam Summary & Common Traps

**✅ Correct patterns**

- Put team-shared servers in project-level `.mcp.json`; keep personal experiments in user scope.
- Make `command`, `args`, and `env` explicit and reviewable; keep server names stable.
- Reference secrets with `${ENV_VAR}` and inject real values through the shell, CI/CD secret store, or deployment platform.
- Validate required environment variables separately in CI/CD; do not assume a developer's local config exists there.

**✗ Common traps**

- Committing API keys, tokens, or private credentials directly in `.mcp.json`.
- Registering a team server only in user config, making it non-reproducible for others and CI.
- Having valid JSON that still lacks `command`, `args`, or required `env` values.
- Ignoring scope and turning a local temporary tool into a project dependency.


### 🧪 Practice Questions

**Q1.** A team wants every developer and CI to use the same MCP server. What is the best configuration pattern?

A) Register it in project-level `.mcp.json` and inject secrets through `${ENV_VAR}` references  
B) Put the command in each developer's shell history  
C) Describe the startup command only in README prose  
D) Hard-code the API key in `.mcp.json` so CI does not need a secret

> **Answer: A**  
> **Explanation:** Project config is shareable and reviewable; secrets must come from environment variables or a secret store.

**Q2.** The server works locally but fails in CI with `ORDERS_API_KEY` missing. What is the most likely fix?

A) Add `ORDERS_API_KEY` to the CI/CD secret store and inject it into the job environment  
B) Commit the real key to the repository so all environments match  
C) Change the server scope to `user` so CI inherits the developer's machine config  
D) Remove the `env` field and let the server guess the key

> **Answer: A**  
> **Explanation:** CI does not inherit a developer's local shell; required secrets must be configured explicitly.
